# Controlling a robot (Kamibot) with a LangChain Agent - step-by-step tutorial

Using the **latest LangChain `create_agent`** and the robot library
**[`pykamilab`](https://pypi.org/project/pykamilab/)**, this document explains -
**starting from the easiest step** - how to control the educational robot
**Kamibot (KamibotPi)** with natural-language commands.

> Final goal: have the agent understand a single line of natural language such as
> `"Go forward 10 cm, then turn right, beep, and draw a triangle"`
> and, on its own, pick and call the **move / sound / shape-drawing** tools.

The overall flow looks like this:

```
User natural-language command
      |
      v
 [ LangChain Agent ]  --(the LLM decides which tool to use)--+
      |                                                      |
      v                                                      v
 [ tool function ]  --(calls pykamilab)-->  [ the real Kamibot robot ]
```

## Step 1. Installation

Create a virtual environment and install the required packages.

In [2]:
!pip install -U langchain "langchain[openai]" pykamilab

## Step 2. Setting the API key

Do not hardcode the key in your code; keep it in an **environment variable**.

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = "sk-..."

## Step 3. First try the robot library on its own (without the agent)

Before attaching an agent, first confirm that the robot moves using **`pykamilab` alone**.
Once you reach this point, all that's left is "wrapping these functions as tools".

### 3-1. Find the port
- Windows: `COM3`, `COM8` ...
- macOS: `/dev/cu.usbserial-*`
- Linux: `/dev/ttyUSB*` or `/dev/ttyACM*`


In [4]:
import pykamilab
pykamilab.list_ports()   # print the list of connectable ports


PORT           DESCRIPTION
-------------- ----------------------------------------
COM1           Communication Port (COM1)
COM69          USB Serial Port(COM69)


[('COM1', 'Communication Port (COM1)'), ('COM69', 'USB Serial Port(COM69)')]

## Step 4. The smallest agent: a single tool (forward only)

Using the same `create_agent` pattern seen in
[`langchain-simple-agent.md`](langchain-simple-agent.md), let's first build an agent with
**just one tool**. This is the step where you get a feel for the core idea.

What happens internally:

1. The user says `"Go forward 20 centimeters"`
2. The agent **decides on its own** that it should use the `move_forward` tool
3. It runs the tool with `distance_cm=20` -> `bot.move_forward_unit(20, "-l")`
4. The robot moves forward, and the agent takes the tool result and generates a natural-language reply

> **Key point:** all we did was wrap a `pykamilab` function with `@tool` and pass it to `create_agent`.
> The rest (which tool to call and when) is handled by the LLM.


In [8]:
from langchain.agents import create_agent
from langchain.tools import tool
from pykamilab import KamibotPi

bot = KamibotPi("COM69", response_timeout=3.0)
bot.init()

KamibotPi Connect PORT=COM69, BAUD=57600


In [9]:
# The function's docstring becomes the 'tool description', which the LLM uses to decide when to call it.
@tool
def move_forward(distance_cm: int) -> str:
    """Moves the robot forward by the given distance (cm)."""
    bot.move_forward_unit(distance_cm, "-l")
    return f"Moved forward {distance_cm} cm."


agent = create_agent(
    model="openai:gpt-5.4",          # format: "<provider>:<model>"
    tools=[move_forward],
    system_prompt="You are an assistant that controls a robot.",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "Go forward 5 centimeters"}]}
)
print(result["messages"][-1].content)


Moved forward 5 cm.


## Step 5. Expand to four movement tools (forward / backward / left / right)

Now you just add more tools. Let's create **forward, backward, left, and right**.

In [10]:
@tool
def move_forward(distance_cm: int) -> str:
    """Moves the robot forward by the given distance (cm)."""
    bot.move_forward_unit(distance_cm, "-l")
    return f"Moved forward {distance_cm} cm."


@tool
def move_backward(distance_cm: int) -> str:
    """Moves the robot backward by the given distance (cm)."""
    bot.move_backward_unit(distance_cm, "-l")
    return f"Moved backward {distance_cm} cm."


@tool
def turn_left(degree: int = 90) -> str:
    """Turns the robot in place to the left by the given angle (degrees)."""
    bot.turn_left_speed(degree, speed=100)
    return f"Turned left {degree} degrees."


@tool
def turn_right(degree: int = 90) -> str:
    """Turns the robot in place to the right by the given angle (degrees)."""
    bot.turn_right_speed(degree, speed=100)
    return f"Turned right {degree} degrees."




In [ ]:
agent = create_agent(
    model="openai:gpt-5.4",          # format: "<provider>:<model>"
    tools=[move_forward, move_backward, turn_left, turn_right],
    system_prompt="You are an assistant that controls a robot. Run multiple actions in order.",
)

agent.invoke({"messages": [{"role": "user",
    "content": "Go forward 10 cm, turn left 90 degrees, then go forward 10 cm again"}]})